# Inference
I use first five percent of train data to predict

In [1]:
import os
import pandas as pd
import torch
import wandb
from datasets import load_dataset, Dataset
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
from huggingface_hub import login

In [2]:
new_model = "etc_on_dd"
base_model = "michellejieli/emotion_text_classifier"
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [3]:
def preprocessing(data):
    data = data.rename_column("utterance", "text")
    data = data.rename_column("emotion", "label")
    data = data.remove_columns(["dialog_id", "turn_type"])
    return data

def shifting_test(data):
    df = data.to_pandas()
    df["label"] = df["label"].shift(-1).fillna(0).astype(int)
    modified_dataset = Dataset.from_pandas(df)
    data = modified_dataset
    return data

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_labels = 7
id2label = {
    0: "neutral",
    1: "anger",
    2: "disgust",
    3: "fear",
    4: "happiness",
    5: "sadness",
    6: "surprise"
}

label2id = {
    "neutral": 0,
    "anger": 1,
    "disgust": 2,
    "fear": 3,
    "happiness": 4,
    "sadness": 5,
    "surprise": 6
}

In [5]:
classifier_model = AutoModelForSequenceClassification.from_pretrained(new_model, num_labels=num_labels, id2label=id2label, label2id=label2id)

In [6]:
classifier_model

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (

In [7]:
classifier = pipeline("sentiment-analysis", model=classifier_model, tokenizer=tokenizer, device=0)
data_name = "benjaminbeilharz/better_daily_dialog"
data = load_dataset(data_name, split='test', num_proc=8)
data = preprocessing(data)
data = shifting_test(data)

In [14]:
data

Dataset({
    features: ['text', 'label'],
    num_rows: 7740
})

In [8]:

def predict(row):
    text = row['text']
    true_label = row['label']
    predicted_result = classifier(text)[0]
    predicted_label = label2id[predicted_result["label"]]

    return {"predicted_label": predicted_label, "true_label": true_label}
predictions = data.map(predict)
true_labels = [p["true_label"] for p in predictions]
predicted_labels = [p["predicted_label"] for p in predictions]

f1 = f1_score(true_labels, predicted_labels, average='weighted')
accuracy = accuracy_score(true_labels, predicted_labels)

print("F1-score:", f1, )
print("Accuracy:", accuracy)

Map:   0%|          | 0/7740 [00:00<?, ? examples/s]

/home/user/.cache/pypoetry/virtualenvs/chat-bot-20tW9agt-py3.11/lib/python3.11/site-packages/transformers/pipelines/base.py:1157: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


F1-score: 0.7503772400462866
Accuracy: 0.8184754521963824


In [9]:
classifier_model = AutoModelForSequenceClassification.from_pretrained(base_model, num_labels=num_labels, id2label=id2label, label2id=label2id)

In [10]:
classifier = pipeline("sentiment-analysis", model=classifier_model, tokenizer=tokenizer, device=0)

In [11]:
def predict(row):
    text = row['text']
    true_label = row['label']
    predicted_result = classifier(text)[0]
    predicted_label = label2id[predicted_result["label"]]

    return {"predicted_label": predicted_label, "true_label": true_label}
predictions = data.map(predict)
true_labels = [p["true_label"] for p in predictions]
predicted_labels = [p["predicted_label"] for p in predictions]

f1 = f1_score(true_labels, predicted_labels, average='weighted')
accuracy = accuracy_score(true_labels, predicted_labels)

print("F1-score:", f1, )
print("Accuracy:", accuracy)

Map:   0%|          | 0/7740 [00:00<?, ? examples/s]

/home/user/.cache/pypoetry/virtualenvs/chat-bot-20tW9agt-py3.11/lib/python3.11/site-packages/transformers/pipelines/base.py:1157: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


F1-score: 0.06624359882391427
Accuracy: 0.11640826873385013


In [13]:
# !jupyter nbconvert --to script inference.ipynb